In [1]:
#%run ./LLM/OpenAIClient

In [42]:
from AzureOpenAIClient import AzureOpenAIClient
from AzureAISearchClient import AzureAISearchClient

In [23]:
# %run 
# ./LLM/AzureAISearchClient

# 認証情報 (Azure OpenAI Service)

In [43]:
openai_api_key = "ASecZYTdvbYZKKwGNiitNIARu7AKQd05G9bYhdLjpDYbwSoV3NdPJQQJ99ALACYeBjFXJ3w3AAABACOG0pBY"
openai_endpoint = "https://eastustago.openai.azure.com/"
openai_api_version = "2024-10-01-preview"

In [25]:
import pandas as pd

In [26]:
# 整形済みデータの読み込み
# df = spark.table("poc2.plasticated_0902abe_b").toPandas()
# display(df)
import pandas as pd
# dfをpickleで読み込む
structured_df = pd.read_pickle(r"./Structured_DF/structured_df.pkl")
display(structured_df.head())
print(len(structured_df))

,filename,title,content,tableinfo,graphinfo,tabledesc,summary,plain_content,graphdesc
Index,,,,,,,,,
0,1_08-39_1章_元.pdf,"[{'id': 5, 'file_name': '1_08-39_1章_元.pdf', 'f...","[{'id': 3, 'file_name': '1_08-39_1章_元.pdf', 'f...","[{'id': 46, 'file_name': '1_08-39_1章_元.pdf', '...","[{'id': 1, 'file_name': '1_08-39_1章_元.pdf', 'f...","[{'id': 46, 'content': 'ファイル名：1_08-39_1章_元\n\n...",ファイル名：1_08-39_1章_元\n\nファイルに書かれている内容：このファイルは、数学...,"文字を使うと,いろいろな数や量を,1つの式で表せたね。\n数や量を表す時に,や0などの記号の...",[]


1


In [27]:
# ID列を付与
structured_df['ID'] = range(1, len(structured_df.index) + 1)
print(structured_df)

               filename                                              title  \
Index                                                                        
0      1_08-39_1章_元.pdf  [{'id': 5, 'file_name': '1_08-39_1章_元.pdf', 'f...   

                                                 content  \
Index                                                      
0      [{'id': 3, 'file_name': '1_08-39_1章_元.pdf', 'f...   

                                               tableinfo  \
Index                                                      
0      [{'id': 46, 'file_name': '1_08-39_1章_元.pdf', '...   

                                               graphinfo  \
Index                                                      
0      [{'id': 1, 'file_name': '1_08-39_1章_元.pdf', 'f...   

                                               tabledesc  \
Index                                                      
0      [{'id': 46, 'content': 'ファイル名：1_08-39_1章_元\n\n...   

                                         

In [ ]:
# 抽象型
# 再作成の際は一旦無視
index_schema = {
    "fields": [
        {"name": "ID", "type": "Edm.String", "key": True, "searchable": False},
        {"name": "Title", "type": "Edm.String", "searchable": True,"analyzer":"ja.lucene"},
        {"name": "MajorCategory", "type": "Edm.String", "searchable": False,"analyzer":"ja.lucene"},
        {"name": "SubCategory", "type": "Edm.String", "searchable": False,"analyzer":"ja.lucene"},
        {"name": "Content", "type": "Edm.String", "searchable": False,"analyzer":"ja.lucene"},
        {"name": "Summary", "type": "Edm.String", "searchable": True,"analyzer":"ja.lucene"},
        {"name": "SummaryVector", "type": "Collection(Edm.Single)", "searchable": True,"retrievable":False,"stored":False,"vectorSearchProfile": "vector-profile-1","dimensions":3072},
        {"name": "TableDesc", "type": "Edm.String", "searchable": False,"analyzer":"ja.lucene"},
    ],
  "semantic": {
    "defaultConfiguration": None,
    "configurations": [
      {
        "name": "q2-semantic_configuration",
        "prioritizedFields": {
          "titleField": {
            "fieldName": "Title"
          },
          "prioritizedContentFields": [
            {
              "fieldName": "Summary",
            }
          ],
        }
      }
    ]
  },
    "vectorSearch": {
        "algorithms": [
            {
                "name": "poc2-hnsw-1",
                "kind": "hnsw",
                "hnswParameters":
                {
                    "m": 4,
                    "efConstruction": 400,
                    "efSearch": 500,
                    "metric": "cosine"
                }
            },
        ],
        "profiles": [
            {
                "name": "vector-profile-1",
                "algorithm": "poc2-hnsw-1"
            }
      ]
    },
}

In [ ]:
# index作成
import traceback
index_name='index-rag-extractive-summary-verify'

#Azure AI Searchの情報を設定
aisearch_service_name = 'pdfsearcher'
aisearch_api_version = "2024-05-01-preview"
aisearch_api_key = "FhhwPvk7exA2g7NaMhpHWRzvYMtDPeMdFMzHVM0BQNAzSeD1W6Jw"
#AzureAISearchClientをインスタンス化
client = AzureAISearchClient(aisearch_service_name, aisearch_api_version, aisearch_api_key)

#インデックスの削除
try:
    client.delete_index(index_name=index_name)
    print("delete index")
#インデックスの作成
    res = client.create_index(index_name=index_name, index_schema=index_schema)
    print(f"create index response:{res}")
except Exception as e:
    print(e)
    traceback.print_stack()


#インデックスの一覧を取得
indexes = client.get_indexes()
print(indexes)
print("作成完了")

Not found index-rag-extractive-summary-verify
delete index
400 Client Error: Bad Request for url: https://pdfsearcher.search.windows.net/indexes/index-rag-extractive-summary-verify?api-version=2024-05-01-preview


  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\banqu\OneDrive\ドキュメント\勉強\RAG構築\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\Users\banqu\OneDrive\ドキュメント\勉強\RAG構築\.venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.start()
  File "c:\Users\banqu\OneDrive\ドキュメント\勉強\RAG構築\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 739, in start
    self.io_loop.start()
  File "c:\Users\banqu\OneDrive\ドキュメント\勉強\RAG構築\.venv\Lib\site-packages\tornado\platform\asyncio.py", line 205, in start
    self.asyncio_loop.run_forever()
  File "C:\Users\banqu\AppData\Local\Programs\Python\Python311\Lib\asyncio\base_events.py", line 607, in run_forever
    self._run_once()
  File "C:\Users\banqu\AppData\Local\Programs\Python\Python311\Lib\asyncio\base_events.py", line 1922, in _run_once
    handle._run()
  File "C:\Users\banqu\App

['azureblob-index', 'azureblob-index2', 'azureblob-index3', 'index-rag-abstract-summary-verify', 'testpdfdatas', 'vector-1031', 'vector-1730285234183', 'vector-1730286676417']
作成完了


In [30]:
# ベクトル化の動作確認
embed_model = "text-embedding-3-large"
api_key = "ASecZYTdvbYZKKwGNiitNIARu7AKQd05G9bYhdLjpDYbwSoV3NdPJQQJ99ALACYeBjFXJ3w3AAABACOG0pBY"
endpoint = "https://eastustago.openai.azure.com/"
api_version = "2024-10-01-preview"
# インスタンス化
aoai = AzureOpenAIClient(api_key, endpoint, api_version)
# ベクトル化
text = "こんちちは"#df.loc[:,"report"][0]
vector = aoai.get_embedding(embed_model, text)

# 内容と次元数確認
print("次元数：", len(vector))


次元数： 3072


In [0]:
# 1654件   16分
# Azure AI Searchにデータをアップロード
embed_model = "text-embedding-3-large"

aoai = AzureOpenAIClient()

#add documents
#Batch your uploads to Azure Search
#インデックスを作成するバッチ処理
#お試しで10件作成してみる
def batch_upload_json_data_to_index(json_data, embed_model, batch_size=100):#batch_size=1000だと書き込み容量制限に引っかかる
    batch_array = []
    count = 0
    batch_counter = 0
    json_data = eval(json_data)
    for i in json_data:
        #print(i)
        count += 1
        batch_array.append(
            {
                "@search.action": "upload",
                "ID": str(i["ID"]),
                "Title": str(i["title"]),
                "MajorCategory": str(i["majorcategory"]),
                "SubCategory": str(i["subcategory"]),
                "Content": str(i["plain_content"]),
                "Summary": str(i["summary"]),
                "SummaryVector": aoai.get_embedding(embed_model, str(i["summary"])),
                "TableDesc": "",
            }
        )

        # limit batches to 1000 records.
        # When the counter hits a number divisible by 1000, the batch is sent.
        if count % batch_size == 0:
            json = client.add_documents(index_name=index_name, documents=batch_array)
            result_list = json['value']
            for i,r in enumerate(result_list):
                if r['statusCode'] >=400:
                    import pprint
                    print(f"batch:{i}")
                    pprint.pprint(r)
            batch_counter += 1
            print(f"Batch sent! - #{batch_counter}")
            batch_array = []
        #break# 本番では削除する
    # This will catch any records left over, when not divisible by 1000
    if len(batch_array) > 0:
        json = client.add_documents(index_name=index_name, documents=batch_array)
        result_list = json['value']
        for i,r in enumerate(result_list):
            if r['statusCode'] >=400:
                import pprint
                print(f"final batch:{i}")
                pprint.pprint(r)
        batch_counter += 1
        print(f"Final batch sent! - #{batch_counter}")
    print("Done!")

#upload the data
json_data = df.to_json(orient='records')

batch_upload_json_data_to_index(json_data, embed_model)


Batch sent! - #1
Batch sent! - #2
Batch sent! - #3
Batch sent! - #4
Batch sent! - #5
Batch sent! - #6
Batch sent! - #7
Batch sent! - #8
Batch sent! - #9
Batch sent! - #10
Batch sent! - #11
Batch sent! - #12
Batch sent! - #13
Batch sent! - #14
Batch sent! - #15
Batch sent! - #16
Final batch sent! - #17
Done!
